# ADB Indonesia — Independent RRP/PCR recovery (96 projects)

**Purpose:** independently search for **both** RRP and PCR for every project ID, including those absent from the original candidate list. This is a *recovery and audit* workflow, not proof that every document exists. It preserves your previous archive and never silently treats a different project suffix as a confirmed match.

**Run cells one by one.** ADB may change its website or restrict access. Stop on HTTP 403/429. The notebook uses public ADB pages, slow requests, no access-bypass techniques. Download results are saved progressively so a Colab disconnection does not lose the audit.

In [ ]:
!pip -q install "requests>=2.31,<3" "beautifulsoup4>=4.12,<5" "pandas>=2,<3"

In [ ]:
import requests, pandas as pd, re, csv, time, json, hashlib, zipfile, io
from pathlib import Path
from urllib.parse import urljoin, urlparse, quote
from bs4 import BeautifulSoup
from google.colab import files
ROOT=Path('/content/ADB_Indonesia_Recovery'); ROOT.mkdir(exist_ok=True)
PDFDIR=ROOT/'PDFs'; PDFDIR.mkdir(exist_ok=True)
SESSION=requests.Session(); SESSION.headers.update({'User-Agent':'Mozilla/5.0 (compatible; academic document research)'})
DELAY=2.5
PROJECT_IDS = ['38354-013', '48207-002', '50297-001', '50395-006', '51114-001', '51157-001', '52282-001', '52316-001', '52332-001', '53211-003', '53220-001', '54152-001', '54224-001', '54224-002', '54256-001', '54428-001', '55020-001', '55063-001', '56257-001', '57074-001', '52282-003', '54461-003', '56017-001', '56140-001', '32507-013', '32507-023', '33409-013', '34100-013', '34152-013', '34159-013', '35139-013', '35182-043', '35183-013', '36541-013', '36541-023', '37049-013', '37049-023', '37475-013', '38117-013', '38264-013', '38385-013', '38479-034', '39071-013', '39597-013', '40003-013', '40009-013', '40009-023', '40009-033', '40009-043', '40061-013', '40247-013', '40272-013', '40273-013', '41074-013', '42099-013', '42109-013', '42362-013', '43009-013', '43092-013', '43211-013', '43220-014', '43251-025', '43255-013', '44252-013', '46093-001', '46093-004', '46094-001', '46239-001', '46421-001', '48134-001', '48134-006', '48134-007', '48207-001', '48207-004', '49043-001', '49043-002', '49043-003', '49080-001', '49141-001', '50016-001', '50168-001', '50168-002', '50168-003', '52218-001', '52218-002', '52324-001', '53211-001', '53211-002', '54117-001', '54139-001', '54461-001', '54461-002', '55235-001', '56368-001', '57018-001', '58165-001']
assert len(PROJECT_IDS)==96 and len(set(PROJECT_IDS))==96
print('Projects:',len(PROJECT_IDS))

## 1. Optional: upload your original approved index and old ZIP

Upload `document_index.csv` and `ADB_Indonesia_RRP_PCR_Downloads.zip` together if you have them. This avoids losing the old matches and permits merging the existing files. If the ZIP is large, upload just the CSV and keep the old archive safely on your laptop. The new recovery ZIP is separate.

In [ ]:
uploaded=files.upload()
for name,blob in uploaded.items():
    if name.lower().endswith('.csv'):
        (ROOT/'previous_document_index.csv').write_bytes(blob)
    elif name.lower().endswith('.zip'):
        (ROOT/'previous_archive.zip').write_bytes(blob)
print('Previous CSV present:',(ROOT/'previous_document_index.csv').exists())
print('Previous ZIP present:',(ROOT/'previous_archive.zip').exists())

## 2. Search both document types independently

Each project is queried on its official ADB project page **and** the ADB document-search endpoint. Candidate links are classified by report type. Discovery results are **not** assumed correct merely because the project number appears in a URL. A single PCR may cover several subprograms.

In [ ]:
def official(u):
    h=(urlparse(u).hostname or '').lower()
    return h=='adb.org' or h.endswith('.adb.org')

def fetch(url,timeout=35):
    if not official(url): raise ValueError('Non-ADB URL')
    r=SESSION.get(url,timeout=timeout)
    if r.status_code in (403,429): raise RuntimeError(f'ACCESS_RESTRICTED {r.status_code} {url}')
    r.raise_for_status()
    return r

def candidates_from_html(url,html,pid):
    soup=BeautifulSoup(html,'html.parser'); out=[]
    for a in soup.select('a[href]'):
        u=urljoin(url,a.get('href',''))
        if not official(u):continue
        path=urlparse(u).path.lower(); title=a.get_text(' ',strip=True)
        if '/projects/documents/' not in path and not path.endswith('.pdf'):continue
        txt=(title+' '+path).lower()
        if 'completion report' in txt or re.search(r'(?:^|[-_/])pcr(?:[-_.]|$)',txt):kind='PCR'
        elif 'recommendation of the president' in txt or re.search(r'(?:^|[-_/])rrp(?:[-_.]|$)',txt):kind='RRP'
        else:continue
        # A search page may include other projects; preserve those only if the project family appears.
        family=pid.split('-')[0]
        if family not in txt:continue
        out.append(dict(project_id=pid,document_type=kind,document_page=u,link_text=title,found_on=url))
    return out

def search_urls(pid,kind):
    # Independent queries for each document type; these endpoints may change.
    phrase='Report and Recommendation of the President' if kind=='RRP' else 'Project Completion Report'
    return [f'https://www.adb.org/projects/{pid}/main',
            'https://www.adb.org/projects/documents?searchstax%5Bquery%5D='+quote(pid+' '+phrase),
            'https://www.adb.org/projects/documents?searchstax%5Bquery%5D='+quote(pid)]

DISCOVERY=ROOT/'independent_discovery.csv'; ERRORS=ROOT/'discovery_errors.csv'
allrows=[]; errors=[]
for i,pid in enumerate(PROJECT_IDS,1):
    for kind in ('RRP','PCR'):
        hits=[]
        for url in search_urls(pid,kind):
            try:
                r=fetch(url)
                hits.extend(x for x in candidates_from_html(r.url,r.text,pid) if x['document_type']==kind)
            except Exception as e:
                errors.append(dict(project_id=pid,document_type=kind,url=url,error=str(e)))
                if 'ACCESS_RESTRICTED' in str(e):
                    pd.DataFrame(allrows).to_csv(DISCOVERY,index=False)
                    pd.DataFrame(errors).to_csv(ERRORS,index=False)
                    raise
            time.sleep(DELAY)
        seen=set()
        for h in hits:
            key=(h['project_id'],h['document_type'],h['document_page'])
            if key not in seen:allrows.append(h);seen.add(key)
    if i%5==0 or i==96:
        pd.DataFrame(allrows,columns=['project_id','document_type','document_page','link_text','found_on']).to_csv(DISCOVERY,index=False)
        pd.DataFrame(errors,columns=['project_id','document_type','url','error']).to_csv(ERRORS,index=False)
        print(f'{i}/96 projects; {len(allrows)} candidate links; {len(errors)} page errors')
print('Independent discovery finished:',DISCOVERY)

## 3. Combine old candidates and new candidates; resolve each document page to a main PDF

The old candidate list is used **only as additional leads**. The notebook checks every PDF URL, classifies it by report type, and flags differing project suffixes for review. Ambiguous or absent main PDFs are not auto-approved.

In [ ]:
old=[]
p=ROOT/'previous_document_index.csv'
if p.exists():
    prior=pd.read_csv(p).fillna('')
    for _,r in prior.iterrows():
        if str(r.get('project_id','')) in PROJECT_IDS and str(r.get('document_type','')).upper() in ('RRP','PCR'):
            u=str(r.get('document_url',''))
            if official(u):old.append(dict(project_id=str(r['project_id']),document_type=str(r['document_type']).upper(),document_page=u,link_text=str(r.get('document_title','')),found_on='previous_index'))
new=pd.read_csv(DISCOVERY).fillna('').to_dict('records') if DISCOVERY.exists() and DISCOVERY.stat().st_size else []
combined=pd.DataFrame(old+new,columns=['project_id','document_type','document_page','link_text','found_on']).drop_duplicates(['project_id','document_type','document_page'])
print('Old leads:',len(old),'Independent leads:',len(new),'Combined:',len(combined))

def main_pdfs(page,kind):
    r=fetch(page,timeout=45)
    if r.content[:5]==b'%PDF-':return [r.url]
    soup=BeautifulSoup(r.text,'html.parser'); matches=[]
    for a in soup.select('a[href]'):
        u=urljoin(r.url,a['href'])
        if not official(u):continue
        name=urlparse(u).path.rsplit('/',1)[-1].lower()
        if kind=='RRP' and re.search(r'-rrp(?:-en)?\.pdf$',name):matches.append(u)
        if kind=='PCR' and re.search(r'-pcr(?:-en)?\.pdf$',name):matches.append(u)
    return list(dict.fromkeys(matches))

resolved=[]
for i,r in enumerate(combined.to_dict('records'),1):
    item=dict(r);item.update(pdf_url='',resolution='',review_flag='',download_status='NOT_ATTEMPTED',error='')
    try:
        matches=main_pdfs(r['document_page'],r['document_type'])
        if len(matches)==1:
            item['pdf_url']=matches[0];item['resolution']='ONE_MAIN_PDF'
            # The same family number is not sufficient to establish subproject coverage.
            ids_found=re.findall(r'\d{5}-\d{3}',r['document_page']+' '+matches[0])
            item['review_flag']='CHECK_PROJECT_SUFFIX_OR_PROGRAM_COVERAGE' if any(x!=r['project_id'] for x in ids_found) else 'CHECK_REPORT_TITLE_AND_LOAN_COVERAGE'
        else:item['resolution']=f'AMBIGUOUS_OR_MISSING_MAIN_PDF_{len(matches)}'
    except Exception as e:
        item['resolution']='PAGE_ERROR';item['error']=str(e)
        if 'ACCESS_RESTRICTED' in str(e):
            resolved.append(item);pd.DataFrame(resolved).to_csv(ROOT/'resolved_candidates.csv',index=False);raise
    resolved.append(item)
    if i%10==0:pd.DataFrame(resolved).to_csv(ROOT/'resolved_candidates.csv',index=False);print(i,'/',len(combined))
    time.sleep(DELAY)
resolved_df=pd.DataFrame(resolved);resolved_df.to_csv(ROOT/'resolved_candidates.csv',index=False)
print(resolved_df['resolution'].value_counts())
files.download(str(ROOT/'resolved_candidates.csv'))

## 4. Review before downloading

Open `resolved_candidates.csv`. For each row, confirm that the document title, project ID, financing phase and loan/subprogram are appropriate. Add a column named **approved** and enter **Yes** only for confirmed matches. Save as `resolved_candidates_approved.csv`, then upload it below. A project may legitimately have no PCR yet. **Do not approve every row automatically.**

In [ ]:
up=files.upload()
assert 'resolved_candidates_approved.csv' in up,'Upload the reviewed file named resolved_candidates_approved.csv'
reviewed=pd.read_csv(io.BytesIO(up['resolved_candidates_approved.csv'])).fillna('')
assert {'project_id','document_type','pdf_url','approved'}.issubset(reviewed.columns)
reviewed=reviewed[reviewed.approved.astype(str).str.strip().str.casefold()=='yes'].copy()
print('Approved PDF rows:',len(reviewed))

## 5. Download approved PDFs, verify, and export a new ZIP

Downloads only reviewed direct PDFs; validates PDF signatures, records SHA-256 hashes and preserves distinct project folders. It does not modify the original archive. Failed rows remain in the report.

In [ ]:
download_log=[]
for i,(_,r) in enumerate(reviewed.iterrows(),1):
    pid=str(r['project_id']);kind=str(r['document_type']).upper();u=str(r['pdf_url'])
    rec=dict(project_id=pid,document_type=kind,pdf_url=u,status='FAILED',file='',sha256='',error='')
    try:
        if not official(u) or not urlparse(u).path.lower().endswith('.pdf'):raise ValueError('Not an official direct PDF URL')
        resp=fetch(u,timeout=120)
        if not resp.content.startswith(b'%PDF-') or len(resp.content)<1000:raise ValueError('Not a valid PDF')
        target_dir=PDFDIR/pid;target_dir.mkdir(exist_ok=True)
        original=urlparse(u).path.rsplit('/',1)[-1]
        target=target_dir/f'{pid}_{kind}_{original}'
        target.write_bytes(resp.content)
        rec.update(status='DOWNLOADED',file=str(target.relative_to(ROOT)),sha256=hashlib.sha256(resp.content).hexdigest())
    except Exception as e:
        rec['error']=str(e)
        if 'ACCESS_RESTRICTED' in str(e):
            download_log.append(rec);pd.DataFrame(download_log).to_csv(ROOT/'recovery_download_results.csv',index=False);raise
    download_log.append(rec)
    pd.DataFrame(download_log).to_csv(ROOT/'recovery_download_results.csv',index=False)
    print(f'{i}/{len(reviewed)} {pid} {kind}: {rec["status"]}')
    time.sleep(DELAY)
print(pd.DataFrame(download_log).status.value_counts())
zipfile_path=ROOT/'ADB_Indonesia_Recovery_Only.zip'
with zipfile.ZipFile(zipfile_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in PDFDIR.rglob('*.pdf'):z.write(p,p.relative_to(ROOT))
    for name in ('independent_discovery.csv','discovery_errors.csv','resolved_candidates.csv','recovery_download_results.csv'):
        f=ROOT/name
        if f.exists():z.write(f,name)
print('Recovery ZIP:',zipfile_path,'Size MB:',round(zipfile_path.stat().st_size/1e6,1))
files.download(str(zipfile_path))

## 6. Coverage audit: 96 × 2

This audit distinguishes downloaded reports from missing or unresolved ones. **Not found** is not the same as **not published**. Confirm the latter only from official project status or reporting evidence.

In [ ]:
log=pd.read_csv(ROOT/'recovery_download_results.csv').fillna('') if (ROOT/'recovery_download_results.csv').exists() else pd.DataFrame(columns=['project_id','document_type','status'])
coverage=[]
for pid in PROJECT_IDS:
    for kind in ('RRP','PCR'):
        subset=log[(log.project_id.astype(str)==pid)&(log.document_type==kind)]
        status='DOWNLOADED' if (subset.status=='DOWNLOADED').any() else ('FAILED' if len(subset) else 'NOT_DOWNLOADED_OR_NOT_LOCATED')
        coverage.append(dict(project_id=pid,document_type=kind,recovery_status=status))
audit=pd.DataFrame(coverage);audit.to_csv(ROOT/'recovery_coverage_192_rows.csv',index=False)
print(audit.recovery_status.value_counts());files.download(str(ROOT/'recovery_coverage_192_rows.csv'))